## Step 1: Installing required library
    - We will need psycopg2-binary for connection with Postgres SQL and running queries 
    - Pandas for dataframe ( stroring data for data manipuitaion, cealning, calcutions etc.)
    - sqlalchemy for creating database connections and interact with databases
    - plotly, seaborn,, matplotlib for visualizations

In [1]:
pip install psycopg2-binary pandas sqlalchemy plotly seaborn matplotlib

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 2.8/2.8 MB 33.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


## Step 2 - Connecting with DB
 - Connecting with Postgres.
 - I have created a fallback logic to load a csv if db connection fails or .env doesnt exists
 - Fallback logic is there to ensure any 3rd person using this notebook and still execute each cell.
 - I have uploaded a clean data that can be used.

In [7]:
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# Load .env if present
load_dotenv(".env", override=True)

DATABASE_URL = os.getenv("DB_URL")

if DATABASE_URL:
    try:
        engine = create_engine(DATABASE_URL)

        with engine.connect() as conn:
            result = conn.execute(text("SELECT COUNT(*) FROM candidate_log"))
            print("Connected! Total rows:", result.fetchone()[0])

        df = pd.read_sql("SELECT * FROM candidate_log", engine)
        print("Loaded data from database")

    except Exception as e:
        print(f"Database connection failed: {e}")

        csv_path = Path.cwd() / "data" / "cleaned_data.csv"
        df = pd.read_csv(csv_path)
        print("Loaded data from CSV fallback")

else:
    csv_path = Path.cwd() / "data" / "cleaned_data.csv"
    df = pd.read_csv(csv_path)
    print("Loaded data from CSV fallback")

Connected! Total rows: 8246901
Loaded data from database


## Step 3.A - Knowing our Data
 - We check numbers of columns there data type
 - We check a sample of data for better understanding the data at hand.

In [9]:
df_sample = df.head()
print(df_sample.shape)
print(df_sample.dtypes)
df_sample.head()

(5, 12)
log_id                          int64
candidate_id                   object
logged_at              datetime64[ns]
subject_id                     object
candidate_status               object
question_display_id            object
activity                       object
question_response              object
all_options                    object
question_language              object
question_section               object
question_type                  object
dtype: object


,log_id,candidate_id,logged_at,subject_id,candidate_status,question_display_id,activity,question_response,all_options,question_language,question_section,question_type
0,934870,MjQxMTAxNzQzNQ==,2025-09-23 13:09:41.894,1,Section 3 Question 12,12,Mark for Review & Next,None,"A,B,C,D",HI,3,MCQ
1,934871,MjIwMTI2MTAxNQ==,2025-09-23 13:07:10.301,1,Section 4 Question 1,1,Auto Save,A,"A,B,C,D",EN,4,MCQ
2,934871,MjIwMTI3MzYxNA==,2025-09-23 16:43:55.807,1,Section 2 Question 5,5,Auto Save,B,"A,B,C,D",EN,2,MCQ
3,934871,MzAxMDA3NjQxMA==,2025-09-23 13:30:07.089,1,Section 4 Question 11,11,Auto Save,B,"A,B,C,D",EN,4,MCQ
4,934871,MzAxMzA1Mzg1OQ==,2025-09-23 09:14:00.241,1,Section 2 Question 9,9,Auto Save,D,"A,B,C,D",HI,2,MCQ


## Step 3.B - Understanding the data.
 -  We already know from the problem statement and above sample data that most of data is system generated data.
 -  System generated data are not use for us, we check its percentage and then remove it.
 -  Section 3.B can only be executed if you are connection with DB and not CSV.

In [6]:
query = """
SELECT 
    activity,
    COUNT(*) as count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct
FROM candidate_log
GROUP BY activity
ORDER BY count DESC
"""
df_activity = pd.read_sql(query, engine)
print(df_activity)

                   activity    count    pct
0                 Auto Save  7482437  90.73
1    Mark for Review & Next   536510   6.51
2  UnMark for Review & Next   227954   2.76
